# MAS Pipeline Debugger

Runs the 5-step label-first MAS pipeline and prints every intermediate artifact so you can
pinpoint where things go wrong:

1. **Plan** — what steps the orchestrator generated (player, task, inputs, outputs)
2. **Step-by-step** — for each step: player used, analysis output, tool results, artifacts saved
3. **Workspace** — full workspace after all steps
4. **Final output** — structured records as a DataFrame


In [1]:
import os
import re
import sys
import json
import logging
sys.path.insert(0, '..')

import pandas as pd
from IPython.display import display, Markdown

from src.config import LLM_PROVIDER, get_model_name
from src.context import create_context
from src.standards import METADATA_STANDARDS
from src.core.schema_factory import SchemaFactory
from src.orchestrator.orchestrator import Orchestrator
from src.experimentutils import (
    read_paper_text,
    load_ground_truth,
    build_study_paper_mapping,
    highlight_numbers_and_tables,
)

# Show INFO logs from the MAS so we can follow execution
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

print(f'Provider : {LLM_PROVIDER}')
print(f'Model    : {get_model_name()}')

Provider : google
Model    : gemini-3.1-flash-lite-preview


In [2]:
# ── Paper & schema selection ──────────────────────────────────────────────────
study_id = 5   # <- change to test a different paper

gt_df   = load_ground_truth()
mapping = build_study_paper_mapping(gt_df)

if study_id not in mapping:
    raise ValueError(f'study_id={study_id} not in mapping. Available: {sorted(mapping.keys())}')

paper_path = mapping[study_id]
print(f'Study# {study_id} → {paper_path.split("/")[-1]}')

standard = METADATA_STANDARDS['wopke_100']
n_fields = len(SchemaFactory()._parse_schema_string(standard))
print(f'Schema   : wopke_100 ({n_fields} fields)')

gt_paper = gt_df[gt_df['Study#'] == study_id].reset_index(drop=True)
print(f'GT rows  : {len(gt_paper)}')

Study# 5 → Li 1999 Interspecific complementary and competitive interactions between intercropped maize and faba bean.md
Schema   : wopke_100 (42 fields)
GT rows  : 3


In [3]:
# ── Output schema & execution context ────────────────────────────────────────
OutputSchema = SchemaFactory().create_from_standard(
    standard,
    record_class_name='WopkeRecord',
    output_class_name='WopkeOutput',
    records_key='yield_records',
)
print(f'OutputSchema : {OutputSchema.__name__}')

context = create_context(source=paper_path, name=f'study_{study_id}')
raw_text = read_paper_text(paper_path)
print(f'Document     : {len(raw_text):,} chars')

OutputSchema : WopkeOutput
Document     : 39,523 chars


In [4]:
# ── MAS objective (same as evaluate_3.ipynb) ──────────────────────────────────
mas_objective = f'''You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA RULES**
- Use JSON keys as the EXACT field names in every record.
- Schema descriptions are guidance only — values must be concrete text extracted from the paper.
- Do not rename, add, or remove any schema fields.

**MULTI-RECORD RULE (CRITICAL)**
Each unique combination of crop pair × site × year × treatment level = one SEPARATE record.
Examples:
- 2 density levels × 2 N-treatments = 4 records for the same crop pair.
- Each row in a yield results table is typically a separate record.
Do NOT collapse table rows or merge treatment combinations into a single record.

**YIELD FIELD MAPPING (CRITICAL)**
- `unified yield sc 1` = sole-crop yield of Crop species 1
- `unified yield sc 2` = sole-crop yield of Crop species 2
- `unified yield ic 1` = intercropped yield of Crop species 1
- `unified yield ic 2` = intercropped yield of Crop species 2
Use explicit table headers, row labels, and footnotes to assign yields to the correct species.
If species assignment is ambiguous, set the ambiguous field to null and note the source in `Data source`.
Preserve numeric values exactly — do not round or average.

**SHARED FIELDS**
Fields that are constant across treatments (Year, Lat, Lon, Experimental design, species names,
Intercropping pattern, uniform nutrient inputs) must be filled identically in every record.
Set to null only if genuinely absent from the paper.

**OUTPUT**: One record per unique treatment combination using exact schema field names.
Omit records only if the paper genuinely contains no supporting data.'''

print('Objective built.')

Objective built.


In [5]:
# ── Instantiate orchestrator with the new pipeline topology ───────────────────
orchestrator = Orchestrator(topology_name='pipeline')
print('Orchestrator ready (topology=pipeline, players_per_step=1, debate_rounds=0)')

INFO | PlanExecutor initialized with topology: pipeline
INFO |   Players per step: 1
INFO |   Debate rounds: 0
INFO |   Player pool: ['value_identifier', 'labeller', 'record_grouper', 'record_labeller', 'record_extractor']
INFO | Orchestrator initialized with topology: pipeline


Orchestrator ready (topology=pipeline, players_per_step=1, debate_rounds=0)


In [6]:
# ── Step 1: Generate the plan only (no execution yet) ─────────────────────────
plan = orchestrator.generate_plan(context=context, objective=mas_objective)

if plan is None:
    print('[ERROR] Plan generation failed!')
else:
    display(Markdown('## Generated Plan'))
    for i, step in enumerate(plan.steps):
        display(Markdown(
            f'**Step {i+1}** | player: `{step.player}`  \n'
            f'Task: {step.task}  \n'
            f'Inputs: `{step.inputs}`  \n'
            f'Outputs: `{step.outputs}`'
        ))

INFO | ============================================================
INFO | GENERATING PLAN
INFO | Context: study_5
INFO | Objective: You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:

{
    "Year of data": "Year(s) when the experiment data were collected. Example: 2018 or 2017–2019.",
    "Duration of experiment": "Total time the experiment ran from first sowing to final harvest. Example: 120 days or 2 growing seasons.",
    "Experimental design": "Type of experimental layout used. Example: Randomized Complete Block Design (RCBD).",
    "Sowing date 1": "Date when Crop species 1 (the first species listed in the intercropping system) was sown. Example: Zea mays sown 15 April 2019.",
    "Sowing date 2": "Date when Crop species 2 (the second species listed in th

## Generated Plan

**Step 1** | player: `value_identifier`  
Task: Scan the FULL document and extract every (field, value) pair for every field in the schema. Cover all sections, table rows, figure captions, and footnotes. List each occurrence separately — do NOT collapse rows. Use ONLY the field names defined in meta_analytic_schema.  
Inputs: `{'meta_analytic_schema': 'meta_analytic_schema'}`  
Outputs: `['field_value_pairs']`

**Step 2** | player: `labeller`  
Task: Use the xml_tag_from_field_values tool to wrap each matched value in the document with XML field tags. Do not modify any original text.  
Inputs: `{'field_value_pairs': 'field_value_pairs'}`  
Outputs: `['labeled_text']`

**Step 3** | player: `record_grouper`  
Task: Read the labeled document, identify the experimental design, and produce a COMPLETE list of schema-conformant candidate records — one per unique treatment/measurement combination. Shared fields (location, year, species) are identical across records; treatment-specific fields differ per row. Include record_confidence and treatment_description for each record. Do NOT merge rows into a single record.  
Inputs: `{'labeled_text': 'labeled_text'}`  
Outputs: `['candidate_records']`

**Step 4** | player: `record_labeller`  
Task: Use the xml_tag_records tool to wrap each candidate record's corresponding table row in the labeled document with <Record_N>...</Record_N> tags. This deterministically marks which text evidence belongs to which record. Do not modify any existing XML field tags.  
Inputs: `{'labeled_text': 'labeled_text', 'candidate_records': 'candidate_records'}`  
Outputs: `['record_labeled_text']`

**Step 5** | player: `record_extractor`  
Task: Read the record-labeled document (which has both field XML tags and <Record_N> section markers) and the candidate record scaffold. Verify and correct each candidate record against its tagged evidence section. Produce the final schema-conformant records.  
Inputs: `{'record_labeled_text': 'record_labeled_text', 'candidate_records': 'candidate_records'}`  
Outputs: `['final_meta_analysis_records']`

In [7]:
# ── Step 2: Execute the plan ───────────────────────────────────────────────────
result = orchestrator.execute_plan(
    plan=plan,
    context=context,
    objective=mas_objective,
    output_schema=OutputSchema,
)

print(f'\nSuccess        : {result.success}')
print(f'Steps completed: {result.steps_completed}/{result.plan_steps_count}')
if result.error:
    print(f'Error          : {result.error}')

INFO | ============================================================
INFO | EXECUTING PLAN
INFO | Context: study_5
INFO | Plan steps: 5
INFO | Objective: You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:

{
    "Year of data": "Year(s) when the experiment data were collected. Example: 2018 or 2017–2019.",
    "Duration of experiment": "Total time the experiment ran from first sowing to final harvest. Example: 120 days or 2 growing seasons.",
    "Experimental design": "Type of experimental layout used. Example: Randomized Complete Block Design (RCBD).",
    "Sowing date 1": "Date when Crop species 1 (the first species listed in the intercropping system) was sown. Example: Zea mays sown 15 April 2019.",
    "Sowing date 2": "Date when Crop species 2 (the second 


Success        : True
Steps completed: 5/5


In [8]:
# ── Step 3: Inspect each step result ──────────────────────────────────────────
SEP  = '─' * 80
TRUNC = 2000   # max chars to print per artifact (set None to see full)

def _show(label, text, trunc=TRUNC):
    from pydantic import BaseModel
    if isinstance(text, BaseModel):
        s = text.model_dump_json(indent=2)
    else:
        s = str(text)
    if trunc and len(s) > trunc:
        s = s[:trunc] + f'  ... [{len(s)-trunc} chars truncated]'
    print(f'{label}:\n{s}\n')

for sr in result.step_results:
    print(SEP)
    print(f'STEP {sr.step_index + 1} | player: {sr.player_role} | success: {sr.success}')
    print(f'Task: {sr.task}')
    if sr.error:
        print(f'ERROR: {sr.error}')

    # Individual player outputs
    for pr in sr.individual_results:
        print(f'\n  [{pr.get("player_name", pr.get("player", "?"))}] analysis:')
        _show('  ', pr.get('analysis', ''), trunc=TRUNC)
        if pr.get('tool_results'):
            for tool_name, tool_out in pr['tool_results'].items():
                _show(f'  tool [{tool_name}]', tool_out, trunc=TRUNC)

    # Consolidated (synthesized) result
    if sr.consolidated_result:
        _show('  [synthesized]', sr.consolidated_result, trunc=TRUNC)

    # Artifacts written to workspace
    print(f'  Artifacts produced: {list(sr.artifacts.keys())}')
    for art_name, art_val in sr.artifacts.items():
        _show(f'  artifact [{art_name}]', art_val, trunc=TRUNC)

print(SEP)

────────────────────────────────────────────────────────────────────────────────
STEP 1 | player: value_identifier | success: True
Task: Scan the FULL document and extract every (field, value) pair for every field in the schema. Cover all sections, table rows, figure captions, and footnotes. List each occurrence separately — do NOT collapse rows. Use ONLY the field names defined in meta_analytic_schema.

  [value_identifier] analysis:
  :
[
  ["Year of data", "1997"],
  ["Experimental design", "split-plot"],
  ["Sowing date 1", "maize sown 16 April"],
  ["Sowing date 2", "faba bean sown 27 March"],
  ["Harvest date 1", "maize harvested 20 September"],
  ["Harvest date 2", "faba bean harvested 25 July"],
  ["Lat", "37.0833"],
  ["Lon", "104.6667"],
  ["Crop species 1", "maize"],
  ["Crop species 2", "faba bean"],
  ["Crop type 1", "cereal"],
  ["Crop type 2", "legume"],
  ["Intercropping pattern", "Strip"],
  ["Density ic 1", "6.75 plants m-2"],
  ["Density ic 2", "7.64 plants m-2"],
  

In [9]:
# ── Step 4: Inspect the final workspace ──────────────────────────────────────
display(Markdown('## Final Workspace Keys'))
workspace = result.final_workspace
for key, val in workspace.items():
    v_str = str(val)
    preview = v_str[:300] + ('...' if len(v_str) > 300 else '')
    print(f'  [{key}] ({type(val).__name__}, {len(v_str)} chars): {preview}')
    print()

## Final Workspace Keys

  [initial_objective] (str, 6933 chars): You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:

{
    "Year of data...

  [meta_analytic_schema] (str, 5113 chars): {
    "Year of data": "Year(s) when the experiment data were collected. Example: 2018 or 2017–2019.",
    "Duration of experiment": "Total time the experiment ran from first sowing to final harvest. Example: 120 days or 2 growing seasons.",
    "Experimental design": "Type of experimental layout use...

  [original_document_text] (dict, 40090 chars): {'Li 1999 Interspecific complementary and competitive interactions between intercropped maize and faba bean': '# Interspecific complementary and competitive interactions between intercropped maize and faba bean\n\nLong $\\mathrm { L i } ^ { 1 }$ , Sicun Yan

In [10]:
# ── Step 5: Reconstruct structured output & compare with GT ──────────────────
raw_mas = workspace.get('final_meta_analysis_records', {})

try:
    if isinstance(raw_mas, OutputSchema):
        result_structured = raw_mas
    elif isinstance(raw_mas, dict):
        result_structured = OutputSchema.model_validate(raw_mas)
    else:
        result_structured = None
        print(f'[WARN] Unexpected type: {type(raw_mas)}')
except Exception as e:
    result_structured = None
    print(f'[WARN] Could not validate MAS output: {e}')
    print(f'  raw_mas type : {type(raw_mas)}')
    if isinstance(raw_mas, dict):
        print(f'  raw_mas keys : {list(raw_mas.keys())}')

if result_structured is not None:
    df_mas = pd.DataFrame([r.model_dump() for r in result_structured.yield_records])
    print(f'MAS extracted {len(df_mas)} record(s)  |  GT has {len(gt_paper)} record(s)')
    display(df_mas)
else:
    df_mas = pd.DataFrame()
    print('[ERROR] No structured records produced — check step results above.')

MAS extracted 3 record(s)  |  GT has 3 record(s)


,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1997,None,split-plot,16 April 1997,27 March 1997,20 September 1997,25 July 1997,37.05,104.40,maize,...,None,None,None,None,Table 2,8.282,5.259,8.928,6.508,t ha-1
1,1997,None,split-plot,16 April 1997,27 March 1997,20 September 1997,25 July 1997,37.05,104.40,maize,...,None,None,None,None,Table 2,9.284,5.012,10.886,8.202,t ha-1
2,1997,None,split-plot,27 March 1997,25 March 1997,25 July 1997,5 July 1997,37.05,104.40,faba bean,...,None,None,None,None,Table 3,5.012,3.776,4.784,4.013,t ha-1


In [12]:
# ── Step 6: Compute MAS evaluation score (same logic as evaluate_3) ───────────
from src.experimentutils import (
    field_similarity_score,
    find_crop_swap_pairs,
    match_records_greedy,
    evaluate_method_scores,
)
from src.experimentutils.eval_utils import _is_missing

if df_mas is None or df_mas.empty:
    print('[ERROR] df_mas is empty – run the MAS pipeline cell above first.')
else:
    # Shared evaluation fields = intersection of Wopke schema fields and GT columns
    wopke_field_names = list(SchemaFactory()._parse_schema_string(standard).keys())
    gt_cols_set   = set(gt_paper.columns)
    shared_fields = [f for f in wopke_field_names if f in gt_cols_set]
    print(f'Shared evaluation fields: {len(shared_fields)} / {len(wopke_field_names)}')

    total_records = len(gt_paper)  # denominator to penalize missing records

    eval_cols = [c for c in shared_fields if c in df_mas.columns]
    if not eval_cols:
        print('[mas] no shared columns between MAS output and GT – cannot score.')
    else:
        scores_df, overall_norm, per_field_norm, n_matches = evaluate_method_scores(
            ext_df=df_mas,
            gt_df=gt_paper,
            shared_cols=eval_cols,
            total_records_for_denominator=total_records,
        )

        print(f"[mas] extracted={len(df_mas)}, matched={n_matches}, reference={total_records}, "
              f"fields={len(eval_cols)} -> normalized ROUGE-L = {overall_norm:.3f}")
        display(scores_df)
        print('\nPer-field normalized scores:')
        for k, v in sorted(per_field_norm.items()):
            print(f'  {k}: {v:.3f}')


Shared evaluation fields: 39 / 42
[mas] extracted=3, matched=3, reference=3, fields=39 -> normalized ROUGE-L = 0.637


,rougeL_Year of data,rougeL_Duration of experiment,rougeL_Experimental design,rougeL_Sowing date 1,rougeL_Sowing date 2,rougeL_Harvest date 1,rougeL_Harvest date 2,rougeL_Lat,rougeL_Lon,rougeL_Crop species 1,...,rougeL_K input SC2,rougeL_K input IC1,rougeL_K input IC2,rougeL_K total in IC,rougeL_Data source,rougeL_unified yield sc 1,rougeL_unified yield sc 2,rougeL_unified yield ic 1,rougeL_unified yield ic 2,rougeL_Yield unit
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.999191,0.997516,1.0,...,0.0,0.0,0.0,0.0,0.5,1.0,1.0,0.333333,0.666667,0.0
1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.999191,0.997516,1.0,...,0.0,0.0,0.0,0.0,0.5,1.0,1.0,0.333333,0.666667,0.0
2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.999191,0.997516,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.520000,0.480000,0.0



Per-field normalized scores:
  rougeL_Crop species 1: 1.000
  rougeL_Crop species 2: 1.000
  rougeL_Crop type 1: 1.000
  rougeL_Crop type 2: 1.000
  rougeL_Data source: 0.667
  rougeL_Density ic 1: 0.840
  rougeL_Density ic 2: 0.667
  rougeL_Density sc 1: 1.000
  rougeL_Density sc 2: 0.667
  rougeL_Duration of experiment: 0.000
  rougeL_Experimental design: 1.000
  rougeL_Harvest date 1: 0.000
  rougeL_Harvest date 2: 0.000
  rougeL_Intercropping pattern: 0.000
  rougeL_K input IC1: 0.000
  rougeL_K input IC2: 0.000
  rougeL_K input SC1: 0.000
  rougeL_K input SC2: 0.000
  rougeL_K total in IC: 0.000
  rougeL_Lat: 0.999
  rougeL_Lon: 0.998
  rougeL_N input IC1: 1.000
  rougeL_N input IC2: 1.000
  rougeL_N input SC1: 1.000
  rougeL_N input SC2: 1.000
  rougeL_N total in IC: 1.000
  rougeL_P input IC1: 1.000
  rougeL_P input IC2: 1.000
  rougeL_P input SC1: 1.000
  rougeL_P input SC2: 1.000
  rougeL_P total in IC: 1.000
  rougeL_Sowing date 1: 0.000
  rougeL_Sowing date 2: 0.000
  rouge

In [ ]:
# ── (Optional) Pretty-print a specific workspace artifact in full ─────────────
# Change the key to whichever artifact you want to inspect without truncation.

INSPECT_KEY = 'labeled_text'   # e.g. 'field_value_pairs', 'candidate_records', 'record_labeled_text'

if INSPECT_KEY in workspace:
    val = workspace[INSPECT_KEY]
    print(f'=== {INSPECT_KEY} ({len(str(val))} chars) ===')
    print(val)
else:
    print(f'Key {INSPECT_KEY!r} not found in workspace.')
    print(f'Available keys: {list(workspace.keys())}')